# Practical Exercise Solutions

Solutions 3, 4 and 5 need `customer_churn.csv` uploaded. Every block is self-contained.

## 1 — Exercise 1: your first ML workflow

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

np.random.seed(42)
n = 200
houses = pd.DataFrame({
    "area":      np.random.randint(600, 3000, n),
    "bedrooms":  np.random.randint(1, 5, n),
    "age_years": np.random.randint(0, 30, n),
})
houses["price"] = (2800 * houses["area"] + 150000 * houses["bedrooms"]
                   - 40000 * houses["age_years"] + np.random.normal(0, 300000, n)).round(0)

print(houses.shape)
print(houses.describe().round(0))

# 1. Features and target
X = houses[["area", "bedrooms", "age_years"]]
y = houses["price"]

# 2. Split BEFORE training
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# 3. Train
model = LinearRegression()
model.fit(X_train, y_train)

# 4. Score on both sets
pred = model.predict(X_test)
print("\nTrain R2:", round(model.score(X_train, y_train), 3))
print("Test  R2:", round(r2_score(y_test, pred), 3))
print("Test  MAE: Rs", round(mean_absolute_error(y_test, pred), 0))

# 5. Read the coefficients out loud
print("\nIntercept:", round(model.intercept_, 0))
for feature, coef in zip(X.columns, model.coef_):
    print(f"  {feature:<10} {coef:>12,.0f}")

# 6. Predict a new house
new_house = pd.DataFrame([{"area": 1500, "bedrooms": 3, "age_years": 5}])
print("\nPredicted price: Rs", round(model.predict(new_house)[0], 0))

# INTERPRETATION
# area       ~ +2,864 -> each extra square foot adds about Rs 2,864
# bedrooms   ~ +154,376 -> each extra bedroom adds about Rs 1.54 lakh
# age_years  ~ -39,187 -> each year of age removes about Rs 39,000
# Train and test R2 are close (0.979 vs 0.960) -> the model generalises.

## 2 — Exercise 2: advertising spend to sales

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

np.random.seed(42)
n = 120
ads = pd.DataFrame({
    "tv_spend":      np.random.uniform(5, 100, n).round(1),
    "digital_spend": np.random.uniform(2, 60, n).round(1),
})
ads["sales"] = (25 + 2.1*ads["tv_spend"] + 3.4*ads["digital_spend"]
                + np.random.normal(0, 30, n)).round(1)

print(ads.head())
print(ads.describe().round(1))

# Visualise both relationships
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(ads["tv_spend"], ads["sales"], color="#1F4E79", s=25)
axes[0].set_xlabel("TV spend"); axes[0].set_ylabel("sales")
axes[0].set_title("TV spend vs sales")
axes[1].scatter(ads["digital_spend"], ads["sales"], color="#2E7D57", s=25)
axes[1].set_xlabel("Digital spend"); axes[1].set_ylabel("sales")
axes[1].set_title("Digital spend vs sales")
plt.tight_layout(); plt.show()

# Split, train, predict
X = ads[["tv_spend", "digital_spend"]]
y = ads["sales"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

lin = LinearRegression().fit(X_train, y_train)
pred = lin.predict(X_test)

# All four metrics
mae  = mean_absolute_error(y_test, pred)
mse  = mean_squared_error(y_test, pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, pred)
print(f"\nMAE : {mae:.2f}")
print(f"MSE : {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2  : {r2:.3f}")

# Actual vs predicted
plt.figure(figsize=(6, 5))
plt.scatter(y_test, pred, color="#1F4E79", s=35)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         "r--", lw=2)
plt.xlabel("actual sales"); plt.ylabel("predicted sales")
plt.title("Actual vs Predicted"); plt.tight_layout(); plt.show()

# Coefficients
print("\nIntercept:", round(lin.intercept_, 2))
for f, c in zip(X.columns, lin.coef_):
    print(f"  {f:<15} {c:>7.3f}")

# BUSINESS ANSWER
# Digital returns about 3.07 units of sales per unit spent; TV returns 2.24.
# At the current spend levels, shifting marginal budget from TV to digital
# should raise sales. Caveat: the data only covers TV spend of 5-100 and
# digital of 2-60. This model cannot tell you what happens at digital = 200,
# where saturation almost certainly sets in.

## 3 — Exercise 3: logistic regression for churn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score,
                             classification_report)

df = pd.read_csv("customer_churn.csv")
print(df.shape)
print(df["churn"].value_counts(normalize=True).round(3))

num_cols = ["tenure_months", "monthly_charges", "support_tickets",
            "satisfaction_score", "usage_hours"]
data = df[num_cols + ["churn"]].dropna()
X = data[num_cols]
y = data["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Scale: fit on TRAIN only, then apply to both
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

log = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_s, y_train)

pred  = log.predict(X_test_s)
proba = log.predict_proba(X_test_s)[:, 1]
print("\nFirst 10 probabilities:", proba[:10].round(3))

# Confusion matrix as a heatmap
cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Pred stay", "Pred churn"],
            yticklabels=["Actual stay", "Actual churn"])
plt.title("Confusion matrix - threshold 0.5")
plt.tight_layout(); plt.show()

print("\nAccuracy :", round(accuracy_score(y_test, pred), 3))
print("Precision:", round(precision_score(y_test, pred), 3))
print("Recall   :", round(recall_score(y_test, pred), 3))
print("F1       :", round(f1_score(y_test, pred), 3))
print("ROC-AUC  :", round(roc_auc_score(y_test, proba), 3))
print("\n", classification_report(y_test, pred, target_names=["Stayed", "Churned"]))

# Coefficients: which feature pushes churn up hardest?
coefs = pd.Series(log.coef_[0], index=num_cols).sort_values()
print(coefs.round(3))

# Lower the threshold to catch more churners
pred_035 = (proba >= 0.35).astype(int)
print("\nThreshold 0.50 -> recall", round(recall_score(y_test, pred), 3),
      "| precision", round(precision_score(y_test, pred), 3))
print("Threshold 0.35 -> recall", round(recall_score(y_test, pred_035), 3),
      "| precision", round(precision_score(y_test, pred_035), 3))

cm2 = confusion_matrix(y_test, pred_035)
print("\nExtra churners caught:", cm2[1, 1] - cm[1, 1])
print("Extra wasted contacts:", cm2[0, 1] - cm[0, 1])

# BUSINESS ANSWER
# tenure_months has the strongest negative coefficient: every additional month
# as a customer sharply reduces churn probability. support_tickets is the
# strongest positive driver.
# Dropping the threshold from 0.50 to 0.35 catches 8 more real churners at the
# cost of 23 extra calls. At Rs 18,000 lifetime value versus Rs 700 per contact,
# 8 saved customers are worth far more than 23 wasted calls - so the lower
# threshold is clearly the right business choice here.

## 4 — Exercise 4: decision tree and the depth dial

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score)

df = pd.read_csv("customer_churn.csv")
num_cols = ["tenure_months", "monthly_charges", "support_tickets",
            "satisfaction_score", "usage_hours"]
data = df[num_cols + ["churn"]].dropna()
X, y = data[num_cols], data["churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Unconstrained tree - watch it memorise
full = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
print("UNLIMITED DEPTH")
print("  train:", round(full.score(X_train, y_train), 3))
print("  test :", round(full.score(X_test,  y_test),  3))
print("  actual depth grown:", full.get_depth())

# Sweep the depth
print("\ndepth   train    test   recall")
for depth in [2, 3, 5, 10, None]:
    t = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_train, y_train)
    print(f"{str(depth):>5}   {t.score(X_train, y_train):.3f}   "
          f"{t.score(X_test, y_test):.3f}   {recall_score(y_test, t.predict(X_test)):.3f}")

# Best tree
best = DecisionTreeClassifier(max_depth=5, min_samples_leaf=20,
                              random_state=42).fit(X_train, y_train)
pred = best.predict(X_test)
print("\nTuned tree (depth 5)")
print("  Accuracy :", round(accuracy_score(y_test, pred), 3))
print("  Precision:", round(precision_score(y_test, pred), 3))
print("  Recall   :", round(recall_score(y_test, pred), 3))
print("  F1       :", round(f1_score(y_test, pred), 3))

# Visualise (depth 3 so the picture stays readable)
show = DecisionTreeClassifier(max_depth=3, min_samples_leaf=25,
                              random_state=42).fit(X_train, y_train)
plt.figure(figsize=(18, 8))
plot_tree(show, feature_names=num_cols, class_names=["Stay", "Churn"],
          filled=True, rounded=True, fontsize=9)
plt.title("Churn decision tree (depth 3)")
plt.tight_layout(); plt.show()

# INTERPRETATION
# The unlimited tree hits 1.000 on training and drops to about 0.70 on test:
# textbook overfitting. Depth 5 scores roughly 0.79 on both.
# Root question: satisfaction_score <= 7.05 - the tree considers dissatisfaction
# the single most informative signal.
# Two rules for the retention team, read straight off the diagram:
#   IF satisfaction <= 7.05 AND tenure <= 18.5 months -> HIGH RISK, call this week
#   IF satisfaction >  7.05 AND tenure >  18.5 months -> LOW RISK, no action

## 5 — Exercise 5: random forest vs single tree

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)

df = pd.read_csv("customer_churn.csv")
num_cols = ["tenure_months", "monthly_charges", "support_tickets",
            "satisfaction_score", "usage_hours"]
data = df[num_cols + ["churn"]].dropna()
X, y = data[num_cols], data["churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

def score_model(name, model, scale=False):
    if scale:
        sc = StandardScaler().fit(X_train)
        model.fit(sc.transform(X_train), y_train)
        pred  = model.predict(sc.transform(X_test))
        proba = model.predict_proba(sc.transform(X_test))[:, 1]
    else:
        model.fit(X_train, y_train)
        pred  = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]
    return {"Model": name,
            "Accuracy":  accuracy_score(y_test, pred),
            "Precision": precision_score(y_test, pred),
            "Recall":    recall_score(y_test, pred),
            "F1":        f1_score(y_test, pred),
            "ROC-AUC":   roc_auc_score(y_test, proba)}

rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42)
results = pd.DataFrame([
    score_model("Logistic Regression",
                LogisticRegression(max_iter=1000, random_state=42), scale=True),
    score_model("Decision Tree (d=5)",
                DecisionTreeClassifier(max_depth=5, random_state=42)),
    score_model("Random Forest", rf),
]).round(3)
print(results.to_string(index=False))

# Feature importance
imp = pd.Series(rf.feature_importances_, index=num_cols).sort_values()
plt.figure(figsize=(8, 4))
imp.plot(kind="barh", color="#1F4E79")
plt.title("Random Forest feature importance")
plt.xlabel("importance"); plt.tight_layout(); plt.show()
print("\n", imp.sort_values(ascending=False).round(3))

# Does more trees help?
print("\nn_estimators   accuracy   recall")
for n_trees in [10, 50, 300]:
    m = RandomForestClassifier(n_estimators=n_trees, max_depth=8,
                               random_state=42).fit(X_train, y_train)
    p = m.predict(X_test)
    print(f"{n_trees:>12}   {accuracy_score(y_test, p):.3f}   {recall_score(y_test, p):.3f}")

# INTERPRETATION
# The forest does NOT beat Logistic Regression here. That is the real lesson:
# this data was generated by a roughly linear process, so the linear model is
# already close to the ceiling and the ensemble has nothing extra to find.
# Top features: tenure_months and satisfaction_score.
# Causality: neither is safely causal. Low satisfaction and churn plausibly
# share an upstream cause (a bad service experience). To test it you would need
# an experiment - randomly intervene on a subset and compare churn rates.
# More trees is not automatically better: 50 trees scores slightly higher than
# 300 here. Beyond roughly 50 the metrics simply fluctuate with noise.

## 6 — Exercise 6: customer segmentation with K-Means

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

np.random.seed(42)
n = 300
mall = pd.DataFrame({
    "age":           np.random.randint(18, 70, n),
    "annual_income": np.random.randint(150000, 2500000, n),
})
mall["spending_score"] = np.clip(
    100 - mall["annual_income"]/30000 + np.random.normal(0, 18, n), 1, 100).round(0)

print(mall.describe().round(0))

feats = ["age", "annual_income", "spending_score"]

# Scaling is mandatory: income is in lakhs, age is in tens
X_scaled = StandardScaler().fit_transform(mall[feats])

# Elbow
inertias = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(range(1, 11), inertias, "o-", color="#1F4E79", lw=2)
plt.xlabel("K"); plt.ylabel("inertia")
plt.title("Elbow curve"); plt.grid(True); plt.tight_layout(); plt.show()

# Fit the chosen K
K = 4
kmeans = KMeans(n_clusters=K, n_init=10, random_state=42)
mall["cluster"] = kmeans.fit_predict(X_scaled)

# Visualise on the two most business-relevant axes
plt.figure(figsize=(8, 5))
plt.scatter(mall["annual_income"], mall["spending_score"],
            c=mall["cluster"], cmap="Set1", s=40)
plt.xlabel("annual income"); plt.ylabel("spending score")
plt.title(f"Customer segments (K={K})")
plt.tight_layout(); plt.show()

# Profile the segments - THIS is where clusters become segments
profile = mall.groupby("cluster")[feats].mean().round(0)
profile["customers"] = mall["cluster"].value_counts().sort_index()
print(profile)

# BUSINESS NAMING AND ACTIONS
# (your cluster numbers may differ - read the profile table, not the numbers)
# Cluster with young + low income + high spending  -> "Young Enthusiastic Spenders"
#     Action: referral programme and app-first offers; they engage cheaply.
# Cluster with older + low income + high spending  -> "Loyal Value Seekers"
#     Action: loyalty points and bundle discounts.
# Cluster with older + high income + low spending  -> "Affluent Low-Engagement"
#     Action: premium concierge onboarding call - highest upside per customer.
# Cluster with young + high income + low spending  -> "Untapped Professionals"
#     Action: premium product trials and lifestyle campaigns.
#
# NOTE: K-Means returned four mathematical groups. Every name and every action
# above was supplied by a human reading the profile table.